# ⚖️ Notebook 04 — Modelo Comparativo: KNN vs. Decision Tree

Se compara KNN con un **Árbol de Decisión (Decision Tree)** como modelo de referencia.  
El árbol de decisión es paramétrico, interpretable y no requiere normalización, lo que lo hace
un contraste teórico interesante para KNN.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, roc_auc_score, roc_curve)
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import label_binarize

sns.set_theme(style='whitegrid')
class_names = ['Low', 'Medium', 'High']
print('Librerías cargadas ✔')

In [ ]:
# ── Cargar datos procesados ───────────────────────────────────────────────
X_train = np.load('data/processed/X_train_sc.npy')
X_test  = np.load('data/processed/X_test_sc.npy')
y_train = np.load('data/processed/y_train.npy')
y_test  = np.load('data/processed/y_test.npy')

with open('data/processed/knn_model.pkl', 'rb') as f:
    knn_final = pickle.load(f)

best_k = knn_final.n_neighbors
print(f'KNN cargado (K={best_k}) | Train: {X_train.shape} | Test: {X_test.shape}')

## 1. Entrenamiento del Modelo Comparativo — Decision Tree

In [ ]:
# Búsqueda de max_depth óptimo con CV
depth_range = range(2, 21)
dt_cv_scores = []

print('Evaluando profundidades del árbol...')
for d in depth_range:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42, class_weight='balanced')
    scores = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
    dt_cv_scores.append(scores.mean())

best_depth = list(depth_range)[np.argmax(dt_cv_scores)]
print(f'\n✔ Mejor profundidad: {best_depth} (CV Acc={max(dt_cv_scores):.4f})')

# Entrenar con mejor profundidad
dt_final = DecisionTreeClassifier(max_depth=best_depth, random_state=42, class_weight='balanced')
dt_final.fit(X_train, y_train)
y_pred_dt = dt_final.predict(X_test)
print('Árbol de Decisión entrenado ✔')

## 2. Predicciones de KNN (ya entrenado)

In [ ]:
y_pred_knn = knn_final.predict(X_test)

# Métricas KNN
acc_knn  = accuracy_score(y_test, y_pred_knn)
f1w_knn  = f1_score(y_test, y_pred_knn, average='weighted')
f1m_knn  = f1_score(y_test, y_pred_knn, average='macro')

# Métricas DT
acc_dt   = accuracy_score(y_test, y_pred_dt)
f1w_dt   = f1_score(y_test, y_pred_dt, average='weighted')
f1m_dt   = f1_score(y_test, y_pred_dt, average='macro')

# AUC
y_test_bin = label_binarize(y_test, classes=[0,1,2])
auc_knn = roc_auc_score(y_test_bin, knn_final.predict_proba(X_test), average='macro', multi_class='ovr')
auc_dt  = roc_auc_score(y_test_bin, dt_final.predict_proba(X_test),  average='macro', multi_class='ovr')

print('='*60)
print(f'{'Métrica':<25} {'KNN':>10} {'Decision Tree':>15}')
print('='*60)
print(f'{'Accuracy':<25} {acc_knn:>10.4f} {acc_dt:>15.4f}')
print(f'{'F1-Score (weighted)':<25} {f1w_knn:>10.4f} {f1w_dt:>15.4f}')
print(f'{'F1-Score (macro)':<25} {f1m_knn:>10.4f} {f1m_dt:>15.4f}')
print(f'{'AUC-ROC (macro)':<25} {auc_knn:>10.4f} {auc_dt:>15.4f}')
print('='*60)

## 3. Visualizaciones Comparativas

In [ ]:
# ── Barras comparativas de métricas ───────────────────────────────────────
metrics_labels = ['Accuracy', 'F1 Weighted', 'F1 Macro', 'AUC-ROC']
knn_vals = [acc_knn, f1w_knn, f1m_knn, auc_knn]
dt_vals  = [acc_dt,  f1w_dt,  f1m_dt,  auc_dt]

x = np.arange(len(metrics_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, knn_vals, width, label=f'KNN (K={best_k})', color='steelblue', edgecolor='white')
bars2 = ax.bar(x + width/2, dt_vals,  width, label=f'Decision Tree (depth={best_depth})', color='tomato', edgecolor='white')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold', color='steelblue')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=10, fontweight='bold', color='tomato')

ax.set_xticks(x)
ax.set_xticklabels(metrics_labels, fontsize=12)
ax.set_ylim(0.5, 1.05)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparación de Métricas: KNN vs. Decision Tree', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('docs/figs/model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Matrices de confusión lado a lado ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(axes,
                               [y_pred_knn, y_pred_dt],
                               [f'KNN (K={best_k})', f'Decision Tree (depth={best_depth})']):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=class_names, yticklabels=class_names, linewidths=0.5)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('Real')
    ax.set_xlabel('Predicho')

plt.suptitle('Matrices de Confusión — Comparación', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/figs/comparison_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Curvas ROC comparativas ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
colors_roc = ['#4CAF50', '#FF9800', '#F44336']

for ax, model, y_pred, title in zip(
    axes,
    [knn_final, dt_final],
    [y_pred_knn, y_pred_dt],
    [f'KNN (K={best_k})', f'Decision Tree (depth={best_depth})']):

    y_proba = model.predict_proba(X_test)
    aucs = []
    for i, (cls, col) in enumerate(zip(class_names, colors_roc)):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
        auc = roc_auc_score(y_test_bin[:, i], y_proba[:, i])
        aucs.append(auc)
        ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC={auc:.3f})')
    ax.plot([0,1],[0,1],'k--', lw=1)
    ax.set_title(f'{title}\nAUC macro={np.mean(aucs):.3f}', fontweight='bold')
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.legend(fontsize=9)

plt.suptitle('Curvas ROC — KNN vs. Decision Tree', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('docs/figs/comparison_roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Importancia de features (Decision Tree) ───────────────────────────────
feature_names = ['Major_Category', 'Year_of_Study', 'Pre_Semester_GPA',
                  'Weekly_GenAI_Hours', 'Primary_Use_Case', 'Prompt_Engineering_Skill',
                  'Tool_Diversity', 'Paid_Subscription', 'Traditional_Study_Hours',
                  'Perceived_AI_Dependency', 'Institutional_Policy',
                  'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 'Skill_Retention_Score']

importances = pd.Series(dt_final.feature_importances_, index=feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 7))
importances.plot(kind='barh', ax=ax, color='tomato', edgecolor='white')
ax.set_title('Importancia de Features — Decision Tree', fontsize=13, fontweight='bold')
ax.set_xlabel('Importancia (Gini)')
plt.tight_layout()
plt.savefig('docs/figs/dt_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nNota: Las features más importantes en el árbol también son las más discriminativas para KNN.')

## 4. Análisis Comparativo — Tabla Resumen

| Criterio | KNN | Decision Tree |
|---|---|---|
| **Tipo** | No paramétrico, lazy | Paramétrico, eager |
| **Requiere normalización** | ✔ Sí (crítico) | ✗ No |
| **Interpretabilidad** | Baja | Alta (visualizable) |
| **Velocidad de entrenamiento** | O(1) | O(n·d·log n) |
| **Velocidad de predicción** | O(n·d) — lento | O(log n) — rápido |
| **Sensible al ruido** | Alta (K pequeño) | Media |
| **Maldición dimensionalidad** | Alta | Baja |
| **Fronteras de decisión** | No lineales | Rectangulares (axis-aligned) |

### Conclusión del Análisis Comparativo
Ambos modelos presentan resultados comparables en este dataset.  
KNN es más flexible para capturar fronteras no lineales, mientras que el árbol de decisión
ofrece mayor interpretabilidad y velocidad en predicción.  
Para producción con 50,000 registros, el árbol sería más eficiente; para exploración y precisión, KNN es competitivo.